# Dịch EN→VI trên GPU miễn phí của Kaggle

Notebook này biến Kaggle thành **server dịch có URL public** để hệ thống `envi-stream-translator`
chạy nhanh (NMT từ ~3s/câu trên CPU xuống ~0.2s/câu trên GPU T4).

## Chuẩn bị (làm 1 lần, ~3 phút)

1. **Upload code**: Kaggle → **Datasets** → **New Dataset** → kéo thả file
   `dist/envi-stream-translator-kaggle.zip` (194 KB, không chứa model) → **Create**.
2. Trong notebook này, panel **Settings** bên phải:
   - **Accelerator** → `GPU T4 x2`
   - **Internet** → `ON`  ← **bắt buộc** (để pip, tải model, và mở tunnel)
3. Panel **Data** (hoặc *Add Input*) → chọn dataset vừa tạo.
4. Chạy **Run All** (hoặc chạy lần lượt 2 cell bên dưới).

## Kết quả

Script in ra một URL dạng `https://<tên>.trycloudflare.com`:

| Đường dẫn | Dùng để |
|---|---|
| `<URL>/m` | **Nói tiếng Anh** (mic chạy được vì là `https`) |
| `<URL>/room/<sid>` | Người nghe xem phụ đề / quét QR |
| `<URL>/metrics` | Đo độ trễ thực tế (p50/p90) |

**Muốn máy bạn nhẹ đi** (chỉ gửi TEXT lên GPU, audio không rời máy):
```powershell
set NMT_REMOTE_URL=https://<URL>
python desktop_app_v2.py
```

> ⚠️ Giữ cell cuối **đang chạy** trong lúc trình diễn. Phiên Kaggle kéo dài ~9–12 giờ.
> Đóng/Stop là server tắt.


---
## Cell 1 — Kiểm tra GPU + Internet + dataset

Chạy cell này trước. Nếu thấy ❌ thì bật lại theo hướng dẫn ở trên rồi chạy tiếp.

In [ ]:
import os, subprocess, sys

print("=" * 62)
try:
    import torch
    if torch.cuda.is_available():
        print("GPU      :", torch.cuda.get_device_name(0), "| torch", torch.__version__)
    else:
        print("GPU      : KHONG CO  -> Settings > Accelerator > GPU T4 x2")
except Exception as e:
    print("torch    : chua co (", e, ") - se cai o cell sau")

code = subprocess.run("curl -s -m 8 -o /dev/null -w '%{http_code}' https://pypi.org",
                      shell=True, capture_output=True, text=True).stdout.strip()
print("Internet :", "OK" if code in ("200", "301", "302") else
      "KHONG CO -> Settings > Internet > ON (can xac minh so dien thoai)")

print("Input    :", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "-")
print("=" * 62)


---
## Cell 2 — Chạy server + mở URL public

Cell này tự: cài thư viện → tìm code trong dataset → tải model → chạy server → mở tunnel.
**Lần đầu mất 3–6 phút** (tải model). Sau đó nó in ra URL của bạn.

Giữ cell này đang chạy trong lúc demo. Bấm ⏹ Stop để tắt.

In [ ]:
import glob, subprocess, sys

cands = (glob.glob("/kaggle/input/**/scripts/colab_kaggle_server.py", recursive=True)
         + glob.glob("/kaggle/working/**/scripts/colab_kaggle_server.py", recursive=True))

if not cands:
    print("KHONG tim thay code. Hay lam buoc 1 va 3:")
    print("  - Datasets > New Dataset > upload dist/envi-stream-translator-kaggle.zip")
    print("  - Add Input > chon dataset do")
else:
    print("Dung script:", cands[0])
    print("-" * 62)
    subprocess.run([sys.executable, cands[0]])


---
## Xử lý sự cố

| Lỗi | Cách sửa |
|---|---|
| `KHONG thay GPU` | Settings → Accelerator → **GPU T4 x2** → Run All lại |
| `Internet: KHONG CO` | Settings → **Internet → ON** (cần xác minh số điện thoại ở trang Kaggle) |
| `KHONG tim thay code` | Chưa **Add Input** dataset, hoặc dataset không chứa thư mục `envi-stream-translator` |
| Không lấy được URL tunnel | Chạy lại cell 2; nếu vẫn lỗi, thử `pip install pycloudflared` |
| Phiên bị ngắt | Kaggle giới hạn ~9–12h/phiên — chạy lại cell 2 |

### Ghi chú về chi phí

Kaggle cho **30 giờ GPU/tuần miễn phí**, và **chỉ trừ khi cell đang chạy**.
Đóng notebook = không tốn gì. Không có rủi ro hoá đơn như thuê pod theo giờ.
